# 01 — Carga, limpieza y partición del corpus FakeDeS

**Objetivo:** Unir los archivos originales de FakeDeS, aplicar limpieza básica y
generar los splits `train / val / test` con proporciones **80 / 10 / 10**,
estratificados por clase.

| Split | Proporción | Archivo de salida |
|-------|-----------|-------------------|
| Train | 80 %      | `data/processed/train.csv` |
| Val   | 10 %      | `data/processed/val.csv`   |
| Test  | 10 %      | `data/processed/test.csv`  |

**Columnas de entrada relevantes:** `Text` (texto de la noticia), `Category` (etiqueta `true` / `fake`)  
**Semilla fija:** `random_state = 42`

---
## 0 · Imports y rutas base

In [1]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

# ── Rutas del proyecto (el notebook vive en notebooks/, subimos un nivel) ──
ROOT_DIR       = Path('..').resolve()
RAW_DIR        = ROOT_DIR / 'data' / 'raw'
PROCESSED_DIR  = ROOT_DIR / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ── Parámetros de partición ────────────────────────────────────────────────
RANDOM_STATE  = 42          # semilla reproducible
TEST_SIZE     = 0.20        # 20 % → temp  (luego se parte 50/50 → val + test)
VAL_TEST_SIZE = 0.50        # 50 % de temp → val  |  50 % de temp → test
LABEL_COL     = 'Category'  # columna de etiqueta en el Excel original
TEXT_COL      = 'Text'      # columna de texto

print(f'ROOT_DIR      : {ROOT_DIR}')
print(f'RAW_DIR       : {RAW_DIR}')
print(f'PROCESSED_DIR : {PROCESSED_DIR}')

ROOT_DIR      : C:\Users\sebas\Documents\UPC 2026-01\TP1\Desarrollo\fakenews-detector-es
RAW_DIR       : C:\Users\sebas\Documents\UPC 2026-01\TP1\Desarrollo\fakenews-detector-es\data\raw
PROCESSED_DIR : C:\Users\sebas\Documents\UPC 2026-01\TP1\Desarrollo\fakenews-detector-es\data\processed


---
## 1 · Carga del corpus

FakeDeS se distribuye en dos archivos Excel:
- `train.xlsx` — conjunto de entrenamiento original del corpus
- `development.xlsx` — conjunto de desarrollo (validation) original del corpus

Los unimos en un único DataFrame para repartirlos a nuestra conveniencia.

In [2]:
# ── Carga individual ────────────────────────────────────────────────────────
df_train_raw = pd.read_excel(RAW_DIR / 'train.xlsx')
df_dev_raw   = pd.read_excel(RAW_DIR / 'development.xlsx')

print(f'train.xlsx        : {len(df_train_raw):,} filas  |  columnas: {df_train_raw.columns.tolist()}')
print(f'development.xlsx  : {len(df_dev_raw):,} filas  |  columnas: {df_dev_raw.columns.tolist()}')

train.xlsx        : 676 filas  |  columnas: ['Id', 'Category', 'Topic', 'Source', 'Headline', 'Text', 'Link']
development.xlsx  : 295 filas  |  columnas: ['Id', 'Category', 'Topic', 'Source', 'Headline', 'Text', 'Link']


In [3]:
# ── Unión de ambos archivos ─────────────────────────────────────────────────
df = pd.concat([df_train_raw, df_dev_raw], ignore_index=True)
print(f'Total tras unir   : {len(df):,} filas')
print()

# ── Distribución de clases en el corpus completo ────────────────────────────
conteo_clases = df[LABEL_COL].value_counts()
pct_clases    = df[LABEL_COL].value_counts(normalize=True).mul(100).round(2)

print('Distribución de clases (corpus completo):')
print(pd.DataFrame({'Conteo': conteo_clases, 'Porcentaje (%)': pct_clases}).to_string())

Total tras unir   : 971 filas

Distribución de clases (corpus completo):
          Conteo  Porcentaje (%)
Category                        
True         491           50.57
Fake         480           49.43


In [4]:
# ── Vista previa del DataFrame unido ───────────────────────────────────────
df.head(3)

,Id,Category,Topic,Source,Headline,Text,Link
0,1,Fake,Education,El Ruinaversal,"RAE INCLUIRÁ LA PALABRA ""LADY"" EN EL DICCIONAR...","RAE INCLUIRÁ LA PALABRA ""LADY"" EN EL DICCIONAR...",http://www.elruinaversal.com/2017/06/10/rae-in...
1,2,Fake,Education,Hay noticia,"La palabra ""haiga"", aceptada por la RAE","La palabra ""haiga"", aceptada por la RAE La Rea...",https://haynoticia.es/la-palabra-haiga-aceptad...
2,3,Fake,Education,El Ruinaversal,YORDI ROSADO ESCRIBIRÁ Y DISEÑARÁ LOS NUEVOS L...,YORDI ROSADO ESCRIBIRÁ Y DISEÑARÁ LOS NUEVOS L...,http://www.elruinaversal.com/2018/05/06/yordi-...


---
## 2 · Limpieza básica

Pasos aplicados (en orden):
1. **Eliminar filas sin texto** — `Text` vacío o `NaN`
2. **Eliminar duplicados exactos** por columna `Text`
3. **Codificar etiqueta** → `Category`: `fake → 1`, `true → 0`

In [5]:
n_inicial = len(df)

# ── Paso 1: eliminar filas con texto vacío o NaN ────────────────────────────
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL].notna() & (df[TEXT_COL] != '') & (df[TEXT_COL] != 'nan')]
n_tras_vacios = len(df)
print(f'Filas eliminadas por texto vacío/NaN  : {n_inicial - n_tras_vacios:,}')

# ── Paso 2: eliminar duplicados exactos por texto ───────────────────────────
df = df.drop_duplicates(subset=[TEXT_COL])
n_tras_dupes = len(df)
print(f'Filas eliminadas por duplicado exacto : {n_tras_vacios - n_tras_dupes:,}')

print(f'\nRegistros finales tras limpieza        : {n_tras_dupes:,}')

Filas eliminadas por texto vacío/NaN  : 0
Filas eliminadas por duplicado exacto : 5

Registros finales tras limpieza        : 966


In [6]:
# ── Paso 3: codificar etiqueta a binario ───────────────────────────────────
# Normalizamos a minúsculas para absorber variaciones de capitalización
df[LABEL_COL] = df[LABEL_COL].astype(str).str.lower().str.strip()

# Verificar que solo existan los valores esperados
valores_unicos = df[LABEL_COL].unique()
print(f'Valores únicos en "{LABEL_COL}" antes de codificar: {sorted(valores_unicos)}')

valores_validos = {'fake', 'true'}
valores_inesperados = set(valores_unicos) - valores_validos
if valores_inesperados:
    print(f'Valores inesperados encontrados: {valores_inesperados}')
    print('   → Se eliminarán del dataset.')
    df = df[df[LABEL_COL].isin(valores_validos)]

# Codificación: fake → 1, true → 0
df[LABEL_COL] = df[LABEL_COL].map({'fake': 1, 'true': 0})

print()
print(f'Codificación aplicada: fake → 1, true → 0')
print(f'Distribución tras codificar:')
print(df[LABEL_COL].value_counts().rename({1: 'fake (1)', 0: 'true (0)'}).to_string())

# Resetear índice para un DataFrame limpio
df = df.reset_index(drop=True)

Valores únicos en "Category" antes de codificar: ['fake', 'true']

Codificación aplicada: fake → 1, true → 0
Distribución tras codificar:
Category
true (0)    489
fake (1)    477


---
## 3 · Partición estratificada

Estrategia de dos pasos para obtener **80 / 10 / 10** con estratificación:

```
corpus  (100 %)
   ├─ train  (80 %)   ← primera partición (test_size=0.20)
   └─ temp   (20 %)
         ├─ val   (10 %)  ← segunda partición (test_size=0.50)
         └─ test  (10 %)
```

Ambas particiones usan `stratify=Category` para preservar la proporción de clases.

In [7]:
# ── Primera partición: train (80 %) vs temp (20 %) ─────────────────────────
df_train, df_temp = train_test_split(
    df,
    test_size=TEST_SIZE,
    stratify=df[LABEL_COL],
    random_state=RANDOM_STATE,
)

# ── Segunda partición: val (50 % de temp) vs test (50 % de temp) ────────────
df_val, df_test = train_test_split(
    df_temp,
    test_size=VAL_TEST_SIZE,
    stratify=df_temp[LABEL_COL],
    random_state=RANDOM_STATE,
)

# ── Resetear índices ────────────────────────────────────────────────────────
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f'Tamaños obtenidos:')
print(f'  train : {len(df_train):,}  ({len(df_train)/len(df)*100:.1f} %)')
print(f'  val   : {len(df_val):,}   ({len(df_val)/len(df)*100:.1f} %)')
print(f'  test  : {len(df_test):,}   ({len(df_test)/len(df)*100:.1f} %)')
print(f'  TOTAL : {len(df_train)+len(df_val)+len(df_test):,}')

Tamaños obtenidos:
  train : 772  (79.9 %)
  val   : 97   (10.0 %)
  test  : 97   (10.0 %)
  TOTAL : 966


In [8]:
# ── Verificar proporción fake/true en cada split ───────────────────────────
def resumen_split(nombre: str, df_split: pd.DataFrame) -> dict:
    """Devuelve un dict con estadísticas de un split."""
    total   = len(df_split)
    n_fake  = int((df_split[LABEL_COL] == 1).sum())
    n_true  = int((df_split[LABEL_COL] == 0).sum())
    pct_fake = round(n_fake / total * 100, 2)
    pct_true = round(n_true / total * 100, 2)
    return {
        'Split'       : nombre,
        'Total'       : total,
        'fake (1)'    : n_fake,
        'true (0)'    : n_true,
        '% fake'      : pct_fake,
        '% true'      : pct_true,
    }

resumen = pd.DataFrame([
    resumen_split('train', df_train),
    resumen_split('val',   df_val),
    resumen_split('test',  df_test),
    resumen_split('TOTAL', df),
]).set_index('Split')

print('Distribución de clases por split:')
print(resumen.to_string())
print()
print('✅ La columna "% fake" debe ser aproximadamente igual en los tres splits.')

Distribución de clases por split:
       Total  fake (1)  true (0)  % fake  % true
Split                                           
train    772       381       391   49.35   50.65
val       97        48        49   49.48   50.52
test      97        48        49   49.48   50.52
TOTAL    966       477       489   49.38   50.62

✅ La columna "% fake" debe ser aproximadamente igual en los tres splits.


---
## 4 · Guardado de splits y reporte de trazabilidad

Se generan cuatro archivos en `data/processed/`:
- `train.csv`, `val.csv`, `test.csv` — splits listos para fine-tuning
- `trazabilidad_split.json` — metadatos reproducibles del proceso de partición

In [9]:
# ── Guardar splits como CSV ─────────────────────────────────────────────────
df_train.to_csv(PROCESSED_DIR / 'train.csv', index=False, encoding='utf-8')
df_val.to_csv(  PROCESSED_DIR / 'val.csv',   index=False, encoding='utf-8')
df_test.to_csv( PROCESSED_DIR / 'test.csv',  index=False, encoding='utf-8')

print('Archivos guardados:')
for nombre in ['train.csv', 'val.csv', 'test.csv']:
    ruta = PROCESSED_DIR / nombre
    print(f'  {ruta}  ({ruta.stat().st_size / 1024:.1f} KB)')

Archivos guardados:
  C:\Users\sebas\Documents\UPC 2026-01\TP1\Desarrollo\fakenews-detector-es\data\processed\train.csv  (1961.1 KB)
  C:\Users\sebas\Documents\UPC 2026-01\TP1\Desarrollo\fakenews-detector-es\data\processed\val.csv  (241.7 KB)
  C:\Users\sebas\Documents\UPC 2026-01\TP1\Desarrollo\fakenews-detector-es\data\processed\test.csv  (260.9 KB)


In [10]:
# ── Calcular hash MD5 del test set para garantizar reproducibilidad ─────────
# Se hashea el contenido bytes del CSV ya escrito en disco
with open(PROCESSED_DIR / 'test.csv', 'rb') as f:
    hash_md5_test = hashlib.md5(f.read()).hexdigest()

print(f'Hash MD5 de test.csv : {hash_md5_test}')

Hash MD5 de test.csv : 6ac66ee1821ece0669c685dce514631d


In [11]:
# ── Construir reporte de trazabilidad ───────────────────────────────────────
trazabilidad = {
    'fecha'                : datetime.now(timezone.utc).isoformat(),
    'semilla'              : RANDOM_STATE,
    'total_registros'      : len(df),
    'registros_train'      : len(df_train),
    'registros_val'        : len(df_val),
    'registros_test'       : len(df_test),
    'proporcion_fake_train': round((df_train[LABEL_COL] == 1).mean(), 6),
    'proporcion_fake_val'  : round((df_val[LABEL_COL]   == 1).mean(), 6),
    'proporcion_fake_test' : round((df_test[LABEL_COL]  == 1).mean(), 6),
    'hash_md5_test'        : hash_md5_test,
    'fuentes_raw'          : ['train.xlsx', 'development.xlsx'],
    'columna_texto'        : TEXT_COL,
    'columna_etiqueta'     : LABEL_COL,
    'codificacion_etiqueta': {'fake': 1, 'true': 0},
    'eliminados_texto_vacio': n_inicial - n_tras_vacios,
    'eliminados_duplicados' : n_tras_vacios - n_tras_dupes,
}

# ── Guardar JSON ────────────────────────────────────────────────────────────
ruta_trazabilidad = PROCESSED_DIR / 'trazabilidad_split.json'
with open(ruta_trazabilidad, 'w', encoding='utf-8') as f:
    json.dump(trazabilidad, f, ensure_ascii=False, indent=2)

print(f'Reporte guardado en: {ruta_trazabilidad}')
print()
print(json.dumps(trazabilidad, ensure_ascii=False, indent=2))

Reporte guardado en: C:\Users\sebas\Documents\UPC 2026-01\TP1\Desarrollo\fakenews-detector-es\data\processed\trazabilidad_split.json

{
  "fecha": "2026-06-02T00:35:51.784647+00:00",
  "semilla": 42,
  "total_registros": 966,
  "registros_train": 772,
  "registros_val": 97,
  "registros_test": 97,
  "proporcion_fake_train": 0.493523,
  "proporcion_fake_val": 0.494845,
  "proporcion_fake_test": 0.494845,
  "hash_md5_test": "6ac66ee1821ece0669c685dce514631d",
  "fuentes_raw": [
    "train.xlsx",
    "development.xlsx"
  ],
  "columna_texto": "Text",
  "columna_etiqueta": "Category",
  "codificacion_etiqueta": {
    "fake": 1,
    "true": 0
  },
  "eliminados_texto_vacio": 0,
  "eliminados_duplicados": 5
}


---
## 5 · Verificación final

Tabla resumen y hash MD5 del test set para confirmar que todo quedó
correctamente generado y es reproducible.

In [12]:
# ── Tabla resumen: conteos por clase y partición ────────────────────────────
splits = {'train': df_train, 'val': df_val, 'test': df_test}

filas = []
for nombre, df_split in splits.items():
    n_fake = int((df_split[LABEL_COL] == 1).sum())
    n_true = int((df_split[LABEL_COL] == 0).sum())
    total  = len(df_split)
    filas.append({
        'Split'      : nombre,
        'fake (1)'   : n_fake,
        'true (0)'   : n_true,
        'Total'      : total,
        '% fake'     : f"{n_fake/total*100:.2f} %",
        '% true'     : f"{n_true/total*100:.2f} %",
        '% del total': f"{total/len(df)*100:.1f} %",
    })

tabla_resumen = pd.DataFrame(filas).set_index('Split')
print('=== RESUMEN FINAL DE SPLITS ===')
print(tabla_resumen.to_string())

=== RESUMEN FINAL DE SPLITS ===
       fake (1)  true (0)  Total   % fake   % true % del total
Split                                                         
train       381       391    772  49.35 %  50.65 %      79.9 %
val          48        49     97  49.48 %  50.52 %      10.0 %
test         48        49     97  49.48 %  50.52 %      10.0 %


In [13]:
# ── Verificar integridad leyendo los CSVs guardados ─────────────────────────
print('Verificación de lectura de archivos guardados:')
for nombre in ['train.csv', 'val.csv', 'test.csv']:
    df_check = pd.read_csv(PROCESSED_DIR / nombre)
    n_fake   = int((df_check[LABEL_COL] == 1).sum())
    n_true   = int((df_check[LABEL_COL] == 0).sum())
    print(f'  {nombre:<12}  filas={len(df_check):>5,}  '
          f'fake={n_fake:>4,}  true={n_true:>4,}  '
          f'cols={df_check.columns.tolist()}')

Verificación de lectura de archivos guardados:
  train.csv     filas=  772  fake= 381  true= 391  cols=['Id', 'Category', 'Topic', 'Source', 'Headline', 'Text', 'Link']
  val.csv       filas=   97  fake=  48  true=  49  cols=['Id', 'Category', 'Topic', 'Source', 'Headline', 'Text', 'Link']
  test.csv      filas=   97  fake=  48  true=  49  cols=['Id', 'Category', 'Topic', 'Source', 'Headline', 'Text', 'Link']


In [14]:
# ── Hash MD5 del test set (reproducibilidad) ────────────────────────────────
with open(PROCESSED_DIR / 'test.csv', 'rb') as f:
    hash_verificacion = hashlib.md5(f.read()).hexdigest()

coincide = hash_verificacion == hash_md5_test
icono    = '✅' if coincide else '❌'

print(f'{icono} Hash MD5 test.csv : {hash_verificacion}')
print(f'   Registrado en JSON : {hash_md5_test}')
print(f'   Coincide           : {coincide}')
print()
print('🏁 Notebook 01 completado. Los splits están listos en data/processed/')

✅ Hash MD5 test.csv : 6ac66ee1821ece0669c685dce514631d
   Registrado en JSON : 6ac66ee1821ece0669c685dce514631d
   Coincide           : True

🏁 Notebook 01 completado. Los splits están listos en data/processed/
